# Customer Churn Prediction
## Notebook 2 — Data Preprocessing & Feature Engineering

**Goal:** Clean the raw data, create new informative features, and encode everything so it's ready for model training.
**Output:** `data/churn_clean.csv`

### Steps
1. Fix `TotalCharges` — convert to numeric, fill 11 blank values
2. Drop `customerID` and encode the target
3. Encode binary columns — Yes/No → 1/0
4. **Feature engineering** — create `AvgMonthlySpend`, `NumAddOns`, `TenureGroup`
5. One-hot encode multi-category columns
6. Save clean dataset

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("C:/Users/polep/Downloads/Customer_churn/Customer_churn/data/Customer-Churn.csv")
print('Raw shape:', df.shape)

Raw shape: (7043, 21)


## Step 1 — Fix TotalCharges

`TotalCharges` loaded as `object` instead of `float` because 11 rows contain whitespace `' '`.
These belong to customers with `tenure=0` (just signed up), so their total charges are logically 0.

In [3]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

print('TotalCharges dtype:', df['TotalCharges'].dtype)
print('Nulls remaining:', df['TotalCharges'].isnull().sum())

TotalCharges dtype: float64
Nulls remaining: 0


## Step 2 — Drop customerID and Encode Target

In [4]:
# customerID is a random identifier — no predictive value
df = df.drop(columns=['customerID'])

# Encode target: Yes → 1, No → 0
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print('Churn distribution:')
print(df['Churn'].value_counts())

Churn distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64


## Step 3 — Encode Binary Columns

Columns with only Yes/No values (or equivalents like 'No phone service') are mapped to 1/0.

In [6]:
# gender
df['gender'] = df['gender'].map({'Male': 1, 'Female': 0})

# All Yes/No columns
binary_cols = [
    'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling',
    'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies'
]

for col in binary_cols:
    df[col] = df[col].map({
        'Yes': 1,
        'No': 0,
        'No phone service': 0,
        'No internet service': 0
    })

# Fill any remaining NaNs with 0
df = df.fillna(0)

print('Nulls after encoding:', df.isnull().sum().sum())

Nulls after encoding: 0


## Step 4 — Feature Engineering

Encoding only changes how existing information is written. **Feature engineering creates new information** from existing columns. Three new features:

| New feature | How it is built | Why it may help predict churn |
|---|---|---|
| `AvgMonthlySpend` | `TotalCharges / tenure` | What the customer has *actually* paid per month over their lifetime, which can differ from the current `MonthlyCharges` |
| `NumAddOns` | count of security, backup, protection, support and streaming add-ons | Customers with more services are more tied in to the company |
| `TenureGroup` | tenure bucketed into 0–12, 13–36 and 37+ months | New customers churn very differently from loyal ones, and the relationship is not a straight line |

All three are calculated row by row from the same customer's own data, so they cannot leak information from the test set.

In [7]:
# 1. Average monthly spend (tenure = 0 → fall back to current monthly charge)
df['AvgMonthlySpend'] = np.where(
    df['tenure'] > 0,
    df['TotalCharges'] / df['tenure'],
    df['MonthlyCharges']
)

# 2. Number of add-on services
addon_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
              'TechSupport', 'StreamingTV', 'StreamingMovies']
df['NumAddOns'] = df[addon_cols].sum(axis=1)

# 3. Tenure buckets
df['TenureGroup'] = pd.cut(
    df['tenure'],
    bins=[-1, 12, 36, np.inf],
    labels=['0-12 months', '13-36 months', '37+ months']
)

print('New features:')
print(df[['tenure', 'TotalCharges', 'AvgMonthlySpend', 'NumAddOns', 'TenureGroup']].head())

print('\nChurn rate by TenureGroup:')
print((df.groupby('TenureGroup', observed=True)['Churn'].mean() * 100).round(1).astype(str) + '%')

print('\nChurn rate by NumAddOns:')
print((df.groupby('NumAddOns')['Churn'].mean() * 100).round(1).astype(str) + '%')

New features:
   tenure  TotalCharges  AvgMonthlySpend  NumAddOns   TenureGroup
0       1         29.85        29.850000        0.0   0-12 months
1      34       1889.50        55.573529        0.0  13-36 months
2       2        108.15        54.075000        0.0   0-12 months
3      45       1840.75        40.905556        0.0    37+ months
4       2        151.65        75.825000        0.0   0-12 months

Churn rate by TenureGroup:
TenureGroup
0-12 months     47.4%
13-36 months    25.5%
37+ months      11.9%
Name: Churn, dtype: object

Churn rate by NumAddOns:
NumAddOns
0.0    26.5%
Name: Churn, dtype: object


## Step 5 — One-Hot Encode Multi-Category Columns

Columns with 3+ categories can't be mapped to a single number — that would imply a false ordering.
We use one-hot encoding instead: each category becomes its own 0/1 column.

In [10]:
df = pd.get_dummies(
    df,
    columns=['InternetService', 'Contract', 'PaymentMethod', 'TenureGroup'],
    dtype=int
)

print('Final shape:', df.shape)
print('\nAll columns:')
print(df.columns.tolist())
print('\nAny object columns left?', (df.dtypes == 'object').any())

KeyError: "None of [Index(['InternetService', 'Contract', 'PaymentMethod', 'TenureGroup'], dtype='object')] are in the [columns]"

## Step 6 — Save Clean Dataset

In [11]:
df.to_csv('../data/churn_clean.csv', index=False)
print('Saved to data/churn_clean.csv')
print('Shape:', df.shape)
df.head(3)

Saved to data/churn_clean.csv
Shape: (7043, 32)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,...,Contract_Month-to-month,Contract_One year,Contract_Two year,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,TenureGroup_0-12 months,TenureGroup_13-36 months,TenureGroup_37+ months
0,0.0,0,0.0,0.0,1,0.0,0.0,0.0,0.0,0.0,...,1,0,0,0,0,1,0,1,0,0
1,0.0,0,0.0,0.0,34,0.0,0.0,0.0,0.0,0.0,...,0,1,0,0,0,0,1,0,1,0
2,0.0,0,0.0,0.0,2,0.0,0.0,0.0,0.0,0.0,...,1,0,0,0,0,0,1,1,0,0
